# Chapter 28 Guided To-Do Practice Labs

**Book Reference:** Chapter 28, Section 28.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 PyTorch core functions and classes, and pass each **function or class name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch28_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
import sharukh_tutor as tutor

tutor.activate(chapter="ch28", notebook="guided_todos")


---
## To-Do 28.1 (Safe Tensor Conversion): `numpy_to_float_tensor(arr)`

Implement `numpy_to_float_tensor(arr)`:
- Convert a NumPy array `arr` into a detached `torch.float32` tensor copy using `torch.tensor(arr, dtype=torch.float32)`.
- Must NOT share memory with or mutate the input NumPy array (even if `arr` is already `np.float32`).

In [ ]:
def numpy_to_float_tensor(arr):
    """Return a detached torch.float32 copy with no shared memory."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    return torch.tensor(arr, dtype=torch.float32)
    ### END SOLUTION


In [ ]:
tutor.check("ch28_todo_01", numpy_to_float_tensor)


---
## To-Do 28.2 (Scalar Autograd): `autograd_quadratic_grad(x_val)`

Implement `autograd_quadratic_grad(x_val)`:
- Create `x = torch.tensor(float(x_val), requires_grad=True)`.
- Evaluate $y = 3x^2 - 4x + 5$ and call `y.backward()`.
- Return `(round(y.item(), 4), round(x.grad.item(), 4))`.

In [ ]:
def autograd_quadratic_grad(x_val):
    """Compute y = 3x^2 - 4x + 5 and dy/dx via PyTorch autograd."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    x = torch.tensor(float(x_val), requires_grad=True)
    y = 3.0 * x**2 - 4.0 * x + 5.0
    y.backward()
    return (round(y.item(), 4), round(x.grad.item(), 4))
    ### END SOLUTION


In [ ]:
tutor.check("ch28_todo_02", autograd_quadratic_grad)


---
## To-Do 28.3 (Custom Dataset): `ClinicalTabularDataset`

Subclass `torch.utils.data.Dataset`:
- `__init__(self, X, y)`: store `self.X` as `torch.float32` of shape `(n, p)` and `self.y` as `torch.float32` of shape `(n, 1)`.
- `__len__(self)`: return the number of samples `n`.
- `__getitem__(self, idx)`: return `(self.X[idx], self.y[idx])`.

In [ ]:
class ClinicalTabularDataset(Dataset):
    """PyTorch Dataset storing float32 tabular X (n, p) and y (n, 1)."""

    def __init__(self, X, y):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
        ### END SOLUTION

    def __len__(self):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        return len(self.X)
        ### END SOLUTION

    def __getitem__(self, idx):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        return self.X[idx], self.y[idx]
        ### END SOLUTION


In [ ]:
tutor.check("ch28_todo_03", ClinicalTabularDataset)


---
## To-Do 28.4 (Raw-Logit MLP): `ClinicalRiskMLP`

Subclass `nn.Module`:
- `__init__(self, in_dim, hidden_dim=16)`: build `Linear(in_dim, hidden_dim) -> ReLU -> Linear(hidden_dim, 1)`.
- `forward(self, x)`: return raw logits of shape `(batch, 1)` — do **NOT** apply sigmoid inside `forward` because we pair it with `BCEWithLogitsLoss`!

In [ ]:
class ClinicalRiskMLP(nn.Module):
    """Two-layer ReLU MLP outputting raw binary logits (batch, 1)."""

    def __init__(self, in_dim, hidden_dim=16):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        ### END SOLUTION

    def forward(self, x):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        return self.net(x)
        ### END SOLUTION


In [ ]:
tutor.check("ch28_todo_04", ClinicalRiskMLP)


---
## To-Do 28.5 (Canonical 5-Step Training Epoch): `train_one_epoch(model, loader, optimizer, criterion)`

Implement `train_one_epoch(model, loader, optimizer, criterion)`:
- Switch to `model.train()`.
- Loop batches `(xb, yb)` running `optimizer.zero_grad()`, `logits = model(xb)`, `loss = criterion(logits, yb)`, `loss.backward()`, `optimizer.step()`.
- Return the sample-weighted mean epoch loss as a Python `float` rounded to **5 decimal places**.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    """Run one canonical 5-step training epoch and return mean loss."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    model.train()
    total_loss, total_n = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
        total_n += len(xb)
    return round(total_loss / total_n, 5)
    ### END SOLUTION


In [ ]:
tutor.check("ch28_todo_05", train_one_epoch)


---
## To-Do 28.6 (Gradient-Free Evaluation): `evaluate_binary_logits(model, loader)`

Implement `evaluate_binary_logits(model, loader)`:
- Call `model.eval()` and run inside `with torch.no_grad():` using `nn.BCEWithLogitsLoss()`.
- Collect `torch.sigmoid(logits)` probabilities and true binary labels across all batches.
- Return `{"loss": round5, "roc_auc": round4, "accuracy": round4}`.

In [ ]:
def evaluate_binary_logits(model, loader):
    """Evaluate raw-logit binary classifier under eval() + no_grad()."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss, total_n = 0.0, 0
    probs_list, y_list = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(xb)
            total_n += len(xb)
            probs_list.append(torch.sigmoid(logits).cpu().numpy().ravel())
            y_list.append(yb.cpu().numpy().ravel())
    probs = np.concatenate(probs_list)
    y_true = np.concatenate(y_list).astype(int)
    preds = (probs >= 0.5).astype(int)
    return {
        "loss": round(total_loss / total_n, 5),
        "roc_auc": round(float(roc_auc_score(y_true, probs)), 4),
        "accuracy": round(float(np.mean(preds == y_true)), 4),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch28_todo_06", evaluate_binary_logits)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 28.1 (`torch.from_numpy` Aliasing Trap):** When `arr` is already `np.float32`, `torch.from_numpy(arr)` and `torch.as_tensor(arr, dtype=torch.float32)` share the same underlying memory buffer, so later NumPy edits silently corrupt your tensor! `torch.tensor(arr, dtype=torch.float32)` always allocates a clean copy.
- **To-Do 28.4 (Double-Sigmoid Bug):** Putting `nn.Sigmoid()` inside `ClinicalRiskMLP.forward()` and then passing probabilities into `BCEWithLogitsLoss()` applies sigmoid *twice*, clamping predictions into `[0.50, 0.73]` and stalling learning. Always output raw logits from `forward()`.
- **To-Do 28.5 & 28.6 (`zero_grad()` + `eval()`/`no_grad()` Discipline):** Forgetting `optimizer.zero_grad()` accumulates stale gradients across every batch; forgetting `model.eval()` leaves Dropout active during validation; forgetting `torch.no_grad()` wastes memory tracking unneeded evaluation graphs.

In [ ]:
tutor.summary()
